# Clase 03 — Distribuciones empíricas

**Curso:** Estadística Aplicada con Python · **Bloque 1:** Describir los datos
**Tiempo estimado:** 90 minutos · **Capítulo del manual:** `manual/03_distribuciones_empiricas.md`

## Qué vas a ver en esta clase

- Construir e interpretar **histogramas**, **KDE** (un "histograma suavizado"), **diagramas de caja** y **ECDF**.
- Comprobar con un **experimento guiado** cómo el número de *bins* y el ancho de banda pueden cambiar la historia.
- Elegir el gráfico según la pregunta y **comparar grupos** sin engañar (ejes compartidos, sin esconder la forma).
- Tratar de forma explícita los **valores ausentes**.

> **Nota:** la KDE se presenta aquí solo como histograma suavizado. El concepto de **densidad de probabilidad** se formaliza en las clases 4 y 5. No usamos tests ni intervalos de confianza: describimos lo que hay en la muestra.

## Requisitos

- Clases 1 y 2 (población/muestra; media, mediana, cuartiles) y capítulo 06 del curso de *Visualización de Datos con Python*.
- Librerías: `numpy`, `pandas`, `matplotlib`, `seaborn` (ya instaladas en Google Colab; en local: `pip install numpy pandas matplotlib seaborn`).

## Los datos

Dataset `penguins` de `seaborn` (*Palmer Penguins*: Horst, Hill y Gorman): 344 pingüinos de tres especies. Si no hay copia local ni conexión, el notebook genera un dataset **sintético** con las mismas columnas (*ejemplo ilustrativo*).

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla**. Después definimos tres funciones auxiliares: cargar los datos (copia local → descarga → sintético), el generador sintético y `completos`, que **excluye ausentes y dice cuántas filas quita**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid")
print("Librerías cargadas correctamente.")

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Genera un dataset SINTÉTICO con las mismas columnas que `penguins`.

    EJEMPLO ILUSTRATIVO: los valores imitan el orden de magnitud de los datos reales,
    pero NO son pingüinos reales. Solo sirve como alternativa sin conexión."""
    r = np.random.default_rng(semilla)
    # por especie: n, (media, desviación) de cada medida y las islas posibles
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)          # los machos son algo mayores que las hembras
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp,
            "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1),
            "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])),
            "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,     # la masa real viene redondeada a 25 g
            "sex": sexo,
        }))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)               # filas 0 y 1 sin ninguna medida; 9 filas más sin sexo (como en los datos reales)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Carga penguins: copia local -> descarga de seaborn -> sintético."""
    for ruta in ("datos/penguins.csv", "../datos/penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "copia local (" + ruta + ")"
    try:
        return sns.load_dataset("penguins"), "descarga desde seaborn"
    except Exception:
        return generar_penguins_sintetico(), "SINTÉTICO (ejemplo ilustrativo, no son pingüinos reales)"


def completos(df, columnas):
    """Devuelve las filas sin ausentes en `columnas` e INFORMA de cuántas se excluyen."""
    sub = df.dropna(subset=columnas)
    cols = ", ".join(columnas)
    print(f"[{cols}] filas usadas: {len(sub)} de {len(df)}  (excluidas por ausentes: {len(df) - len(sub)})")
    return sub

---
## 1. Los datos y sus valores ausentes

Antes de dibujar nada: ¿de dónde vienen los datos y qué falta? **No los ocultamos**: los contamos.

In [ ]:
df, origen = cargar_penguins()
print("Origen de los datos:", origen)
print(f"Filas: {len(df)}   Columnas: {list(df.columns)}")

faltantes = df.isna().sum()
display(pd.DataFrame({"ausentes": faltantes, "% ausentes": (faltantes / len(df) * 100).round(1)}))
print("Filas con al menos un ausente:", df.isna().any(axis=1).sum())
display(df[df.isna().any(axis=1)].head(6))

**Decisión de tratamiento:** no hacemos un `dropna()` global. Para cada gráfico excluimos solo las filas que **no tienen la variable que dibujamos**, y lo decimos (función `completos`). Así, una fila sin sexo sigue contando en un histograma de masa corporal.

---
## 2. Histograma: contar por intervalos

Primero, a mano y con los doce datos del capítulo. Dividimos el rango en intervalos y contamos cuántos datos caen en cada uno.

In [ ]:
# Los 12 datos de ejemplo del capítulo (ejemplo ilustrativo)
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = [0, 5, 10, 15, 20, 25]      # bordes de los intervalos: [0,5) [5,10) [10,15) [15,20) [20,25]
conteos, _ = np.histogram(x, bins=bordes)
conteos10, _ = np.histogram(x, bins=[0, 10, 20, 30])

print("Conteos con ancho 5 :", conteos)
print("Conteos con ancho 10:", conteos10)
print("Suma de los conteos:", conteos.sum(), "=", len(x))

**Observa:** con ancho 5 se ve un hueco entre 15 y 20 (conteo 0). Con ancho 10 el hueco desaparece: `[10, 20)` contiene 12, 13, 13, 14 y ningún otro. Los mismos datos, dos historias.

Ahora con la masa corporal real de los pingüinos: 5 intervalos calculados por `numpy` y el histograma con 20.

In [ ]:
masa = completos(df, ["body_mass_g"])["body_mass_g"]

# Los mismos pasos con la masa corporal real y 5 intervalos
conteos5, bordes5 = np.histogram(masa, bins=5)
for i in range(5):
    print(f"Intervalo [{bordes5[i]:.0f}, {bordes5[i+1]:.0f}):  conteo = {conteos5[i]}")

fig, ax = plt.subplots()
sns.histplot(masa, bins=20, color="steelblue", ax=ax)
ax.set_title("Masa corporal de los pingüinos (n = " + str(len(masa)) + ")")
ax.set_xlabel("Masa corporal (g)"); ax.set_ylabel("Número de pingüinos")
plt.show()

---
## 3. Experimento guiado: ¿cuántos *bins*? ¿qué ancho de banda?

Tomamos la **longitud de aleta**, una variable que mezcla especies de tamaño distinto. Dibujamos **los mismos datos** con 3, 8, 20 y 60 intervalos, sobre el mismo eje horizontal. Antes de ejecutar, anota: ¿cuántos picos esperas ver?

In [ ]:
aleta = completos(df, ["flipper_length_mm"])["flipper_length_mm"]

fig, ejes = plt.subplots(1, 4, figsize=(16, 3.4), sharex=True)
for ax, k in zip(ejes, [3, 8, 20, 60]):
    sns.histplot(aleta, bins=k, color="steelblue", ax=ax)
    ancho = (aleta.max() - aleta.min()) / k
    ax.set_title(f"{k} intervalos ({'ancho'} ≈ {ancho:.1f} mm)")
    ax.set_xlabel("Longitud de aleta (mm)"); ax.set_ylabel("Número de pingüinos")
plt.tight_layout(); plt.show()

**¿Qué observas?**

- ¿Con cuántos intervalos aparece la separación entre grupos? ¿Con 3 se ve? ¿Con 60 se ve mejor, o aparece ruido?
- ¿Qué característica se mantiene estable en los cuatro gráficos? Esa es la que puedes creerte.
- ¿Qué habría pasado si solo hubieras mirado el primero?

*(Tu respuesta aquí.)*

---
## 4. KDE: el histograma suavizado

La KDE coloca una pequeña campana sobre cada dato y las suma: una curva continua. Aquí es solo un **histograma suavizado**; su significado como densidad de probabilidad llega en las clases 4 y 5. El parámetro clave es el **ancho de banda** (`bw_adjust` en `seaborn`). Probamos tres valores sobre el mismo histograma.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, bw in zip(ejes, [0.2, 1, 3]):
    sns.histplot(aleta, bins=30, stat="density", color="lightsteelblue", ax=ax)
    sns.kdeplot(aleta, bw_adjust=bw, color="crimson", linewidth=2, ax=ax)
    ax.set_title(f"ancho de banda × {bw}")
    ax.set_xlabel("Longitud de aleta (mm)")
ejes[0].set_ylabel("Densidad")
plt.tight_layout(); plt.show()

**¿Qué observas?**

- Con ancho de banda 0,2, ¿cuántos picos dibuja la curva? ¿Son todos "reales"?
- Con 3, ¿qué ha pasado con la separación entre los grupos?
- ¿En qué se parece elegir el ancho de banda a elegir el número de *bins*?

*(Tu respuesta aquí.)*

### Cuidado con los bordes: la KDE se sale del rango de los datos

`seaborn` prolonga la curva unos anchos de banda más allá del mínimo y del máximo (`cut=3` por defecto). Lo medimos con la aleta y lo vemos con una variable que **no puede ser negativa**: unos tiempos de espera simulados *(ejemplo ilustrativo)*.

In [ ]:
# La curva se dibuja más allá del mínimo y del máximo observados
fig, ax = plt.subplots()
sns.kdeplot(aleta, ax=ax)
xs_kde = ax.lines[0].get_xdata()
plt.close(fig)
print("Rango de los datos de aleta        :", aleta.min(), "-", aleta.max())
print("Rango sobre el que dibuja la KDE     :", round(xs_kde.min(), 1), "-", round(xs_kde.max(), 1))

# Tiempos de espera simulados (ejemplo ilustrativo): nunca pueden ser negativos
esperas = rng.exponential(scale=5, size=300)
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
sns.kdeplot(esperas, ax=ejes[0]);           ejes[0].set_title("Espera: KDE por defecto")
sns.kdeplot(esperas, cut=0, ax=ejes[1]);    ejes[1].set_title("Espera: KDE con cut=0")
for ax in ejes:
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Tiempo de espera (min)")
print("Mínimo de los tiempos de espera:", round(esperas.min(), 3))
print("La KDE por defecto llega hasta  :", round(ejes[0].lines[0].get_xdata().min(), 1))
plt.show()

**¿Qué observas?** La KDE por defecto dibuja densidad a la izquierda del 0, donde **no puede haber datos**. No es una predicción: es un efecto del suavizado. Con `cut=0` la curva se corta en el rango observado. Regla: no leas la KDE como "probabilidad de que existan valores" fuera de lo observado.

---
## 5. Diagrama de caja: compacto, pero sin forma

El diagrama de caja resume mediana, cuartiles y valores atípicos (clase 2). Comparamos, para la misma aleta, el diagrama de caja con el histograma.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True)
sns.boxplot(x=aleta, color="lightsteelblue", ax=ejes[0]);   ejes[0].set_title("Diagrama de caja")
sns.histplot(aleta, bins=25, color="steelblue", ax=ejes[1]);  ejes[1].set_title("Histograma de los mismos datos")
for ax in ejes:
    ax.set_xlabel("Longitud de aleta (mm)")
ejes[0].set_ylabel("")
ejes[1].set_ylabel("Número de pingüinos")
plt.tight_layout(); plt.show()
print(aleta.describe().round(1))

**¿Qué observas?** La caja es una sola, con mediana y cuartiles razonables. ¿Se intuye en ella la existencia de dos grupos? El histograma sí la muestra. El diagrama de caja es ideal para **comparar muchos grupos**; pero conviene poner los puntos encima para no perder la forma:

In [ ]:
datos_masa = completos(df, ["body_mass_g", "species"])

fig, ax = plt.subplots(figsize=(8, 3.6))
sns.boxplot(data=datos_masa, x="body_mass_g", y="species", color="lightsteelblue", fliersize=0, ax=ax)
sns.stripplot(data=datos_masa, x="body_mass_g", y="species", color="black", alpha=0.35, size=3, ax=ax)
ax.set_title("Masa corporal por especie"); ax.set_xlabel("Masa corporal (g)"); ax.set_ylabel("")
plt.show()

---
## 6. ECDF: la escalera acumulada

La ECDF en *x* es la proporción de datos ≤ *x*. La construimos a mano con los doce datos y la comparamos con la de `seaborn`.

In [ ]:
# ECDF a mano
x_ord = np.sort(x)
ecdf = np.arange(1, len(x_ord) + 1) / len(x_ord)
print("x   ECDF(x)")
for xv, yv in zip(x_ord, ecdf):
    print(f"{xv:>2}   {yv:.3f}")

masa_por_esp = completos(df, ["body_mass_g", "species"])
fig, ax = plt.subplots()
sns.ecdfplot(data=masa_por_esp, x="body_mass_g", hue="species", ax=ax)
ax.axvline(4000, color="gray", linestyle="--")
ax.set_title("ECDF de la masa corporal por especie"); ax.set_xlabel("Masa corporal (g)"); ax.set_ylabel("Proporción acumulada")
plt.show()

**Lecturas directas:** en el eje vertical se lee la proporción por debajo de un valor (línea discontinua en 4 000 g); en horizontal, el percentil (dónde la curva cruza 0,5 es la mediana). Lo comprobamos con números:

In [ ]:
# Proporción de pingüinos de cada especie con masa ≤ 4000 g
resumen = masa_por_esp.groupby("species")["body_mass_g"].agg(
    **{"proporción ≤ 4000 g": lambda s: (s <= 4000).mean(), "mediana (g)": "median"}
).round(2)
display(resumen)

# Percentiles leídos en la ECDF (cuantiles de la muestra)
print(masa_por_esp["body_mass_g"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

---
## 7. Elegir el gráfico según la pregunta

| Pregunta | Gráfico |
|---|---|
| ¿Qué forma tiene? ¿Uno o varios picos? | Histograma (probando varios *bins*) o KDE |
| ¿Qué proporción está por debajo de un valor? | ECDF |
| ¿Cómo se comparan centro y dispersión de muchos grupos? | Diagrama de caja (con puntos) |
| ¿Cómo difieren las formas de pocos grupos? | KDE o histogramas con **el mismo eje**, más ECDF |

## 8. Comparar grupos sin engañar

Primero, el error clásico: cada panel con su propia escala. Después, la versión honesta: mismos ejes y cada grupo normalizado por separado (`stat="density"`, `common_norm=False`).

In [ ]:
datos_aleta = completos(df, ["flipper_length_mm", "species"])

def panel_especies(compartir, titulo):
    fig, ejes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=compartir, sharey=compartir)
    for ax, (esp, g) in zip(ejes, datos_aleta.groupby("species")):
        sns.histplot(g["flipper_length_mm"], bins=15, stat="density", ax=ax)
        ax.set_title(f"{esp} (n = {len(g)})")
        ax.set_xlabel("Longitud de aleta (mm)"); ax.set_ylabel("Densidad")
    fig.suptitle(titulo)
    plt.tight_layout(); plt.show()

panel_especies(False, "Ejes independientes (engaña)")
panel_especies(True, "Ejes compartidos (honesto)")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 3.6), sharex=True)
sns.kdeplot(data=datos_aleta, x="flipper_length_mm", hue="species", common_norm=False, ax=ejes[0])
ejes[0].set_title("KDE por especie")
sns.ecdfplot(data=datos_aleta, x="flipper_length_mm", hue="species", ax=ejes[1])
ejes[1].set_title("ECDF por especie")
sns.boxplot(data=datos_aleta, x="flipper_length_mm", y="species", ax=ejes[2])
ejes[2].set_title("Caja por especie")
for ax in ejes:
    ax.set_xlabel("Longitud de aleta (mm)")
plt.tight_layout(); plt.show()

resumen_aleta = datos_aleta.groupby("species")["flipper_length_mm"].agg(
    **{"n": "count", "mediana": "median", "Q1": lambda s: s.quantile(0.25), "Q3": lambda s: s.quantile(0.75)}
)
display(resumen_aleta)

### La trampa de "media ± error"

Dos grupos simulados *(ejemplo ilustrativo)*: A con un solo pico y B mezcla de dos poblaciones. Sus medias son casi iguales y sus desviaciones parecidas, así que las barras "media ± desviación" **no distinguen** un grupo del otro.

In [ ]:
# Dos grupos con la misma media y desviación parecida, pero formas muy distintas (ejemplo ilustrativo)
grupo_a = rng.normal(50, 11, size=400)
# B: mezcla de dos poblaciones, centradas en 40 y en 60
grupo_b = np.where(rng.random(400) < 0.5, rng.normal(40, 4, size=400), rng.normal(60, 4, size=400))

for nombre, g in [("A", grupo_a), ("B", grupo_b)]:
    print(f"{nombre}: media = {g.mean():.1f}   desviación = {g.std():.1f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
ejes[0].bar(["Grupo A", "Grupo B"], [grupo_a.mean(), grupo_b.mean()], yerr=[grupo_a.std(), grupo_b.std()], capsize=8, color=["#4c72b0", "#dd8452"])
ejes[0].set_title("Barras: media ± desviación"); ejes[0].set_ylabel("Valor")
sns.histplot(grupo_a, bins=30, alpha=0.6, label="Grupo A", ax=ejes[1])
sns.histplot(grupo_b, bins=30, alpha=0.6, label="Grupo B", ax=ejes[1])
ejes[1].set_title("Distribuciones reales"); ejes[1].set_ylabel("Número de pingüinos"); ejes[1].legend()
plt.tight_layout(); plt.show()

**¿Qué observas?** Las barras de A y B son casi idénticas; los histogramas no lo son en absoluto. Un resumen de dos números no puede reemplazar a la distribución. Y ojo: aquí comparamos **mirando**; decidir si una diferencia es real o ruido requiere los métodos de las clases 6 a 9.

---
## Resumen de la clase

1. **Histograma:** cuenta por intervalos; el número de *bins* cambia la historia, así que prueba varios.
2. **KDE:** un histograma suavizado; el ancho de banda hace el papel de los *bins*, y la curva se sale del rango de los datos (la densidad de probabilidad se formaliza en las clases 4 y 5).
3. **Diagrama de caja:** compacto y bueno para comparar muchos grupos, pero no muestra la forma.
4. **ECDF:** proporción de datos ≤ *x*; sin parámetros que ajustar y ideal para comparar grupos.
5. **Comparar grupos:** mismos ejes, cada grupo normalizado por separado, contar el *n* y los ausentes excluidos, y no sustituir la distribución por "media ± error".

**Siguiente paso:** Clase 4 — Probabilidad para el analista. Y ahora, a practicar: `notebooks_practica/clase03_practica_distribuciones_empiricas.ipynb`.